# CFR and SSI: Evaluating Animal Re-Identification

Standard retrieval metrics show whether a model finds the correct individual, but not which visual cues support the match. This notebook uses [MegaDescriptor-T-224](https://huggingface.co/BVRA/MegaDescriptor-T-224) to evaluate retrieval across four image views: full image, foreground, shortcut, and silhouette.

We compute **Recall@k, random Recall@k, CFR, and SSI** to compare foreground-supported retrieval with identity information available from shortcuts. For the motivation and metric definitions, see the [CFR and SSI guide](../docs/cfr.md). For mask loading and image views, see the [segmentation masks notebook](masks.ipynb).


## 1. Setup

Run this notebook from the `notebooks/` directory. Install WildlifeDatasets from the repository root and add the model dependencies if needed:

```python
%pip install -e ".."
%pip install torch torchvision "timm>=1.0"
```

The example uses up to 20 identities to keep inference manageable. Set `max_identities = None` to evaluate all identities with available masks. The reported scores apply to the selected subset. Dataset files and model weights are downloaded when first needed.


In [ ]:
%pip install torch torchvision "timm>=1.0"

In [7]:
from math import comb

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import recall_score
from tqdm.auto import tqdm

from wildlife_datasets import datasets, metrics

root = "../data/NyalaData"
max_identities = 20
batch_size = 8
retrieval_batch_size = 256
ks = (1, 5, 10)
silhouette_is_shortcut = True  # Nyala identity is carried by coat markings.
view_names = ("full", "foreground", "shortcut", "silhouette")

## 2. Prepare the Evaluation Dataset

Download NyalaData and its segmentation masks using the standard dataset interface. We then keep images with available masks and select the identities to evaluate. This ensures that all four views use the same gallery.

Each image is compared against the other gallery images, excluding itself. Identities represented by only one image can serve as gallery negatives but are excluded from query scoring.


In [8]:
datasets.NyalaData.get_data(root, masks=True)
dataset = datasets.NyalaData(root, remove_unknown=True, factorize_label=True)

DATASET NyalaData: DOWNLOADING STARTED.
You are trying to download an already downloaded dataset.
        This message may have happened to due interrupted download or extract.
        To force the download use the `force=True` keyword such as
        get_data(..., force=True) or download(..., force=True).
        
DATASET NyalaData: FINISHED.



In [9]:
positions = [i for i in range(len(dataset)) if dataset.has_mask(i)]
if not positions:
    raise ValueError("No masks are available. Prepare masks before running inference.")
dataset = dataset.get_subset(positions)

identities = dataset.df[dataset.col_label].drop_duplicates().iloc[:max_identities]
positions = np.flatnonzero(dataset.df[dataset.col_label].isin(identities))
dataset = dataset.get_subset(positions)
labels = dataset.labels
n_images = len(dataset)
counts = pd.Series(labels).value_counts()
positives = pd.Series(labels).map(counts).to_numpy() - 1
valid_queries = positives > 0

if not valid_queries.any():
    raise ValueError("Evaluation needs at least one identity with two masked images.")
if max(ks) >= n_images:
    raise ValueError(f"Choose k < {n_images}: each query has {n_images - 1} gallery images.")
print(f"Gallery: {n_images} images, {len(counts)} identities; scored queries: {valid_queries.sum()}")

Gallery: 357 images, 20 identities; scored queries: 357


## 3. Load the Retrieval Model

We use the pretrained [MegaDescriptor-T-224](https://huggingface.co/BVRA/MegaDescriptor-T-224) model to extract image embeddings. The model requires no additional training for this example. We apply its recommended preprocessing to every image view and run inference on CUDA, Apple MPS, or CPU, depending on availability.


In [10]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0)
model = model.eval().to(device)
data_config = timm.data.resolve_model_data_config(model)
preprocess = timm.data.create_transform(**data_config, is_training=False)
print(f"Device: {device}; model input: {data_config['input_size']}")

Device: mps; model input: [3, 224, 224]


## 4. Extract Image Embeddings

For each image, we generate the full, foreground, shortcut, and silhouette views and apply the same preprocessing. The model produces an embedding for each view. We normalize the embeddings so their dot product gives cosine similarity.


In [11]:
embeddings = {view: [] for view in view_names}
with torch.inference_mode():
    for start in tqdm(range(0, n_images, batch_size), desc="Extracting views"):
        items = [dataset[i].cfr_views for i in range(start, min(start + batch_size, n_images))]
        for view in view_names:
            images = torch.stack([preprocess(item[view]) for item in items]).to(device)
            features = F.normalize(model(images), dim=1)
            embeddings[view].append(features.cpu())

embeddings = {view: torch.cat(batches).numpy() for view, batches in embeddings.items()}
print({view: features.shape for view, features in embeddings.items()})

Extracting views: 100%|██████████| 45/45 [00:36<00:00,  1.24it/s]

{'full': (357, 768), 'foreground': (357, 768), 'shortcut': (357, 768), 'silhouette': (357, 768)}


## 5. Retrieve Matching Images

We rank gallery images by cosine similarity and exclude each query image from its own results. The retrieved image positions are then converted to individual IDs, which we use to compute Recall@k. An individual may appear more than once in the ranking if several gallery images show the same animal.


In [14]:
predictions = {}
max_k = max(ks)
for view, features in embeddings.items():
    ranked_positions = np.empty((n_images, max_k), dtype=int)
    for start in range(0, n_images, retrieval_batch_size):
        stop = min(start + retrieval_batch_size, n_images)
        similarities = features[start:stop] @ features.T
        similarities[np.arange(stop - start), np.arange(start, stop)] = -np.inf
        order = np.argsort(-similarities, axis=1, kind="stable")[:, :max_k]
        ranked_positions[start:stop] = order
    predictions[view] = labels[ranked_positions]

## 6. Compute CFR and SSI

We first calculate Recall@k for each image view. A query counts as correct if at least one of the top-k retrieved images shows the same individual. We also calculate random Recall@k: the probability of finding a matching image by selecting k gallery images at random, without replacement.

CFR and SSI use these recall values and the random baseline to summarize foreground-supported retrieval and recoverable shortcut information. We report all metrics together to make the comparison easier.


In [13]:
rows = []
gallery_size = n_images - 1
for k in ks:
    ranked = {view: values[:, :k] for view, values in predictions.items()}
    recalls = {}
    for view, values in ranked.items():
        hits = (values[valid_queries] == labels[valid_queries, None]).any(axis=1)
        recalls[view] = recall_score(np.ones(valid_queries.sum(), dtype=int), hits, zero_division=0)

    total_rankings = comb(gallery_size, k)
    random_recall = np.mean([
        (total_rankings - comb(gallery_size - int(p), k)) / total_rankings
        for p in positives[valid_queries]
    ])
    rows.append({
        "k": k,
        "Recall@k": recalls["full"],
        "Foreground Recall@k": recalls["foreground"],
        "Shortcut Recall@k": recalls["shortcut"],
        "Silhouette Recall@k": recalls["silhouette"],
        "Random Recall@k": random_recall,
        "CFR": metrics.cfr(recalls, random_recall),
        "SSI": metrics.ssi(recalls, random_recall, silhouette_is_shortcut=silhouette_is_shortcut),
    })

results = pd.DataFrame(rows).set_index("k")
display(results.style.format("{:.1%}", na_rep="undefined"))

,Recall@k,Foreground Recall@k,Shortcut Recall@k,Silhouette Recall@k,Random Recall@k,CFR,SSI
k,,,,,,,
1,42.6%,38.9%,17.9%,12.3%,5.7%,35.2%,33.2%
5,78.4%,74.2%,47.9%,51.0%,25.2%,65.5%,48.4%
10,90.2%,86.0%,67.5%,68.3%,43.7%,75.1%,53.0%


## Interpreting the Results

**High CFR and low SSI** indicate strong retrieval from the animal foreground with little identity information recoverable from shortcut views. **High CFR and high SSI** indicate that both the foreground and shortcut views contain useful identity information. Interpret these scores alongside full-image Recall@k and the random baseline.

For NyalaData, SSI also includes silhouette information. For species where shape is an important identification cue, such as whales identified by their flukes, set `silhouette_is_shortcut=False`. Some scores may be undefined when the random baseline or full-image performance leaves no meaningful signal to compare; these appear as `NaN`.

See the [CFR and SSI guide](../docs/cfr.md) for more on the metrics and their interpretation.
